<a href="https://colab.research.google.com/github/gowrijayesh29-web/Renewable-AI-coordinator-/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [14]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [15]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [16]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
SPORTS = {
    "football": (
        "Team sport"
        "11 players per team"
        "Played with a round ball"
        "Main objective: score goals"
        "Popular competitions: FIFA World Cup UEFA Champions League"
    ),
    "cricket": (
        "Team sport"
        "11 players per team"
        "Played with a bat ball and wickets"
        "Main objective: score more runs than the opposing team"
        "Popular competitions: ICC Cricket World Cup IPL"
    ),
    "basketball": (
        "Team sport"
        "5 players per team on court"
        "Played with a ball and two hoops"
        "Main objective: score points by shooting through the hoop"
        "Popular competitions: NBA FIBA Basketball World Cup"
    ),
    "tennis": (
        "Racket sport"
        "Singles or doubles"
        "Played with rackets a ball and a net"
        "Main objective: win points games and sets"
        "Popular competitions: Wimbledon US Open"
    ),
    "badminton": (
        "Racket sport"
        "Singles or doubles"
        "Played with rackets a shuttlecock and a net"
        "Main objective: win rallies and reach the target score"
        "Popular competitions: BWF World Championships All England Open"
    ),
    "volleyball": (
        "Team sport"
        "6 players per team on court"
        "Played with a ball and a high net"
        "Main objective: ground the ball on the opponent's court"
        "Popular competitions: FIVB World Championship Olympics"
    ),
    "hockey": (
        "Team sport"
        "11 players per team in field hockey"
        "Played with sticks and a ball"
        "Main objective: score goals"
        "Popular competitions: Hockey World Cup Olympics"
    ),
    "baseball": (
        "Team sport"
        "9 players per team"
        "Played with a bat ball gloves and bases"
        "Main objective: score runs by completing bases"
        "Popular competitions: MLB World Series World Baseball Classic"
    ),
    "swimming": (
        "Individual or relay sport"
        "One swimmer or a relay team"
        "Performed in a pool or open water"
        "Main objective: finish the distance in the fastest time"
        "Popular competitions: World Aquatics Championships Olympics"
    ),
    "athletics": (
        "Individual and team sport"
        "Varies by event"
        "Includes running jumping throwing and combined events"
        "Main objective: achieve the fastest time greatest distance or highest score"
        "Popular competitions: World Athletics Championships Olympics"
    ),
    "boxing": (
        "Combat sport"
        "Two competitors"
        "Uses protective gloves in a ring"
        "Main objective: win by points stoppage or knockout"
        "Popular competitions: World Boxing Championships Olympics"
    ),
    "golf": (
        "Individual sport"
        "Individual players or teams"
        "Played with clubs balls and a course of holes"
        "Main objective: complete the course in the fewest strokes"
        "Popular competitions: The Masters The Open Championship"
    ),
    "table_tennis": (
        "Racket sport"
        "Singles or doubles"
        "Played with paddles a lightweight ball and a table net"
        "Main objective: win games by reaching the target score"
        "Popular competitions: ITTF World Championships Olympics"
    ),
    "wrestling": (
        "Combat sport"
        "Two competitors"
        "Contested on a mat using grappling techniques"
        "Main objective: pin the opponent or earn more points"
        "Popular competitions: World Wrestling Championships Olympics"
    ),
    "kabaddi": (
        "Team sport"
        "7 players per team on court"
        "A contact sport with raiding and defending"
        "Main objective: score points through successful raids and tackles"
        "Popular competitions: Pro Kabaddi League Kabaddi World Cup"
    )
}

In [17]:
# @title Define a tool
def search_sports(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in SPORTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_sports("minimum of 4"))


[Source: golf]
Individual sportIndividual players or teamsPlayed with clubs balls and a course of holesMain objective: complete the course in the fewest strokesPopular competitions: The Masters The Open Championship


In [22]:
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="claude",
    tools=[search_sports],
    system_instruction=(
        '''
        You are a knowledgeable sports information assistant.
    When you need specific information about a sport, such as rules,
    player counts, equipment, scoring, tournaments, or gameplay, use
    the search_sports tool.

    Only answer based on information returned by the sports dataset.
    If the dataset does not contain the answer, say so clearly.
    Do not make up facts or use information outside the dataset.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/claude


In [31]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

MODEL_NAME = "gemini-3.8-flash"

# Only defines a placeholder if you haven't already defined search_sports in another cell
if "search_sports" not in globals():
    def search_sports(query: str) -> str:
        return "No data found."

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_sports",
        description="Find out the sport which requires 7 players in the court",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=6):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config,
            )
        except Exception as e:
            transient = "429" in str(e) or "503" in str(e)
            if transient and attempt < max_retries - 1:
                wait = 5 * (attempt + 1)  # 5s, 10s, 15s, 20s, 25s
                print(f"  Server busy, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_sports(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
  Server busy, retrying in 5s...

Final answer:
It looks like your question might be missing some context. 

If this refers to a **specific class, assignment, project, or platform**, please share the details or instructions, and I can give you an exact answer.

In general, however, students are commonly asked to share documents on topics such as:

1. **Coursework & Study Materials**: Lecture summaries, reading notes, study guides, and flashcards.
2. **Collaborative Group Projects**: Research outlines, project proposals, team meeting minutes, and draft reports.
3. **Peer Review & Feedback**: Draft essays, lab reports, code repositories, or creative writing pieces.
4. **Presentations & Seminars**: Slide decks, bibliographies, handouts, and discussion prompts.
5. **Class Discussions & Case Studies**: Responses to weekly discussion prompts, case analyses, or assigned readin

In [32]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Which sport requires more than 2 players",
    # This question is not answered by the database.
    "Can all sports be played outdoors?",
    # This question is part of the general knowledge of the model.
    "Which is the most popular sport?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Which sport requires more than 2 players
--------------------------------------------------
  Server busy, retrying in 5s...
  Server busy, retrying in 10s...
  Server busy, retrying in 15s...
  Server busy, retrying in 20s...
  Server busy, retrying in 25s...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 56.859333232s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.8-flash', 'location': 'global'},

In [33]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)

AssertionError: 